In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q xgboost catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('XGBoost version:', xgb.__version__)
print('Libraries loaded.')

XGBoost version: 3.2.0
Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing — a3 original (best so far) ────────────────────────
def preprocess(df):
    df = df.copy()

    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    # Missing flags before encoding
    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})

    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})

    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # Engineered features
    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)

    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)

    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)

    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    # XGBoost needs no NaN — fill remaining with -1
    df = df.fillna(-1)
    return df


X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 59), X_test: (8834, 59)


In [5]:
# ── Cell 5: Class weights ────────────────────────────────────────────────────
# XGBoost multiclass uses sample_weight (per-row), not class_weights directly
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)
sample_weights = np.array([class_weights[y] for y in y_train])

print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [6]:
# ── Cell 6: XGBoost — 3 seeds x 10 folds ────────────────────────────────────
print('=' * 60)
print('XGBOOST')
print('=' * 60)

xgb_oof_proba  = np.zeros((len(y_train), 10))
xgb_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr,  X_val  = X_train.iloc[tr_idx].values, X_train.iloc[val_idx].values
        y_tr,  y_val  = y_train[tr_idx],             y_train[val_idx]
        sw_tr         = sample_weights[tr_idx]

        dtrain = xgb.DMatrix(X_tr,  label=y_tr,  weight=sw_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        dtest  = xgb.DMatrix(X_test.values)

        params = {
            'objective'        : 'multi:softprob',
            'num_class'        : 10,
            'eval_metric'      : 'mlogloss',
            'learning_rate'    : 0.03,
            'max_depth'        : 6,
            'min_child_weight' : 10,       # equivalent to min_data_in_leaf
            'subsample'        : 0.8,
            'colsample_bytree' : 0.8,
            'reg_alpha'        : 0.1,
            'reg_lambda'       : 1.0,
            'seed'             : SEED,
            'verbosity'        : 0,
            'nthread'          : -1,
        }

        model = xgb.train(
            params,
            dtrain,
            num_boost_round    = 2000,
            evals              = [(dval, 'val')],
            early_stopping_rounds = 100,
            verbose_eval       = False,
        )

        val_proba = model.predict(dval).reshape(-1, 10)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict(dtest).reshape(-1, 10) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    xgb_oof_proba  += oof_proba  / len(SEEDS)
    xgb_test_preds += test_preds / len(SEEDS)

xgb_final_oof = balanced_accuracy_score(y_train, np.argmax(xgb_oof_proba, axis=1))
print(f"\n{'='*60}")
print(f'XGB FINAL OOF BA: {xgb_final_oof:.4f}   (a3 CatBoost baseline: 0.3860)')
print(f"{'='*60}")

XGBOOST

======================================== SEED=42 ========================================
  Fold  1: BA=0.3101  best_iter=499
  Fold  2: BA=0.2708  best_iter=476
  Fold  3: BA=0.3256  best_iter=530
  Fold  4: BA=0.3253  best_iter=576
  Fold  5: BA=0.3414  best_iter=585
  Fold  6: BA=0.3252  best_iter=629
  Fold  7: BA=0.3093  best_iter=483
  Fold  8: BA=0.2674  best_iter=517
  Fold  9: BA=0.3172  best_iter=547
  Fold 10: BA=0.3153  best_iter=477
  OOF BA (seed=42): 0.3106 | mean=0.3108 ± 0.0226

======================================== SEED=7 ========================================
  Fold  1: BA=0.3317  best_iter=578
  Fold  2: BA=0.3114  best_iter=492
  Fold  3: BA=0.3268  best_iter=514
  Fold  4: BA=0.3563  best_iter=608
  Fold  5: BA=0.3002  best_iter=534
  Fold  6: BA=0.3186  best_iter=513
  Fold  7: BA=0.3192  best_iter=572
  Fold  8: BA=0.2596  best_iter=424
  Fold  9: BA=0.3142  best_iter=470
  Fold 10: BA=0.3316  best_iter=488
  OOF BA (seed=7): 0.3168 | mean=0.3170 ±

In [7]:
# ── Cell 7: Try blending XGB + CatBoost ─────────────────────────────────────
# Re-run CatBoost with same a3 preprocessing to get its OOF probabilities
# then blend — if they disagree in different places, averaging helps

print('=' * 60)
print('CATBOOST (a3 baseline, for blending)')
print('=' * 60)

cb_class_counts  = np.bincount(y_train)
cb_class_weights = len(y_train) / (10 * cb_class_counts)

cb_oof_proba  = np.zeros((len(y_train), 10))
cb_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba   = np.zeros((len(y_train), 10))
    test_preds  = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr,  y_val = y_train[tr_idx],      y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            depth                 = 6,
            l2_leaf_reg           = 3,
            class_weights         = cb_class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    cb_oof_proba  += oof_proba  / len(SEEDS)
    cb_test_preds += test_preds / len(SEEDS)

cb_final_oof = balanced_accuracy_score(y_train, np.argmax(cb_oof_proba, axis=1))
print(f'\nCATBOOST FINAL OOF BA: {cb_final_oof:.4f}')

CATBOOST (a3 baseline, for blending)

======================================== SEED=42 ========================================
  Fold  1: BA=0.3824  best_iter=45
  Fold  2: BA=0.4135  best_iter=31
  Fold  3: BA=0.3984  best_iter=26
  Fold  4: BA=0.3770  best_iter=195
  Fold  5: BA=0.3991  best_iter=2
  Fold  6: BA=0.4509  best_iter=161
  Fold  7: BA=0.3566  best_iter=73
  Fold  8: BA=0.3742  best_iter=45
  Fold  9: BA=0.4177  best_iter=49
  Fold 10: BA=0.4252  best_iter=52
  OOF BA (seed=42): 0.3994 | mean=0.3995 ± 0.0267

======================================== SEED=7 ========================================
  Fold  1: BA=0.3803  best_iter=102
  Fold  2: BA=0.4009  best_iter=151
  Fold  3: BA=0.4126  best_iter=139
  Fold  4: BA=0.4125  best_iter=131
  Fold  5: BA=0.3713  best_iter=19
  Fold  6: BA=0.4059  best_iter=80
  Fold  7: BA=0.3734  best_iter=44
  Fold  8: BA=0.3825  best_iter=30
  Fold  9: BA=0.4115  best_iter=287
  Fold 10: BA=0.3849  best_iter=85
  OOF BA (seed=7): 0.3938 

In [8]:
# ── Cell 8: Blend search ─────────────────────────────────────────────────────
print('Blend search (XGBoost weight vs CatBoost weight):')
best_ba, best_w = 0, 0.5

for w_xgb in np.arange(0.0, 1.05, 0.05):
    w_cb    = 1.0 - w_xgb
    blended = w_xgb * xgb_oof_proba + w_cb * cb_oof_proba
    ba      = balanced_accuracy_score(y_train, np.argmax(blended, axis=1))
    marker  = ' ← best' if ba > best_ba else ''
    print(f'  XGB={w_xgb:.2f} / CB={w_cb:.2f}  OOF BA={ba:.4f}{marker}')
    if ba > best_ba:
        best_ba, best_w = ba, w_xgb

print(f'\nBest blend: XGB={best_w:.2f}, CB={1-best_w:.2f}  → OOF BA={best_ba:.4f}')
print(f'XGBoost alone:  {xgb_final_oof:.4f}')
print(f'CatBoost alone: {cb_final_oof:.4f}')
print(f'Best blend:     {best_ba:.4f}')
print(f'a3 baseline:    0.3860')

Blend search (XGBoost weight vs CatBoost weight):
  XGB=0.00 / CB=1.00  OOF BA=0.3933 ← best
  XGB=0.05 / CB=0.95  OOF BA=0.3818
  XGB=0.10 / CB=0.90  OOF BA=0.3739
  XGB=0.15 / CB=0.85  OOF BA=0.3691
  XGB=0.20 / CB=0.80  OOF BA=0.3649
  XGB=0.25 / CB=0.75  OOF BA=0.3642
  XGB=0.30 / CB=0.70  OOF BA=0.3629
  XGB=0.35 / CB=0.65  OOF BA=0.3616
  XGB=0.40 / CB=0.60  OOF BA=0.3585
  XGB=0.45 / CB=0.55  OOF BA=0.3476
  XGB=0.50 / CB=0.50  OOF BA=0.3415
  XGB=0.55 / CB=0.45  OOF BA=0.3415
  XGB=0.60 / CB=0.40  OOF BA=0.3332
  XGB=0.65 / CB=0.35  OOF BA=0.3319
  XGB=0.70 / CB=0.30  OOF BA=0.3304
  XGB=0.75 / CB=0.25  OOF BA=0.3253
  XGB=0.80 / CB=0.20  OOF BA=0.3247
  XGB=0.85 / CB=0.15  OOF BA=0.3218
  XGB=0.90 / CB=0.10  OOF BA=0.3206
  XGB=0.95 / CB=0.05  OOF BA=0.3174
  XGB=1.00 / CB=0.00  OOF BA=0.3163

Best blend: XGB=0.00, CB=1.00  → OOF BA=0.3933
XGBoost alone:  0.3163
CatBoost alone: 0.3933
Best blend:     0.3933
a3 baseline:    0.3860


In [9]:
# ── Cell 9: Per-class recall on best blend ───────────────────────────────────
final_oof_proba = best_w * xgb_oof_proba + (1 - best_w) * cb_oof_proba
oof_labels      = np.argmax(final_oof_proba, axis=1)
final_oof_ba    = balanced_accuracy_score(y_train, oof_labels)

disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
a3_recall = {0:0.314, 1:0.392, 2:0.272, 3:0.351, 4:0.759,
             5:0.338, 6:0.488, 7:0.237, 8:0.571, 9:0.138}

report = classification_report(y_train, oof_labels, output_dict=True)
print(f'Final OOF BA (best blend): {final_oof_ba:.4f}\n')
print(f'{"Class":<5} {"Name":<16} {"a3":>8} {"Now":>8} {"Change":>8}')
print('-' * 52)
for cls in range(10):
    r     = report[str(cls)]['recall']
    r_old = a3_recall[cls]
    delta = r - r_old
    flag  = ' ← up' if delta > 0.02 else ' ← LOW' if r < 0.3 else ''
    print(f'{cls:<5} {disorder_names[cls]:<16} {r_old:>8.3f} {r:>8.3f} {delta:>+8.3f}{flag}')

Final OOF BA (best blend): 0.3933

Class Name                   a3      Now   Change
----------------------------------------------------
0     레베르시                0.314    0.314   -0.000
1     낭포성섬유증              0.392    0.390   -0.002
2     당뇨                  0.272    0.270   -0.002 ← LOW
3     리증후군                0.351    0.348   -0.003
4     암                   0.759    0.793   +0.034 ← up
5     테이-삭스               0.338    0.339   +0.001
6     혈색소침착증              0.488    0.498   +0.010
7     사립체근병종              0.237    0.238   +0.001 ← LOW
8     알츠하이머               0.571    0.604   +0.033 ← up
9     확인안됨                0.138    0.139   +0.001 ← LOW


In [10]:
# ── Cell 10: Save submission ─────────────────────────────────────────────────
final_test_proba = best_w * xgb_test_preds + (1 - best_w) * cb_test_preds
final_preds      = np.argmax(final_test_proba, axis=1)

submission = pd.DataFrame({
    'id'      : TEST_DATA.index,
    'disorder': final_preds
}).set_index('id')
submission.to_csv('submission_xgb_blend.csv')

print('Saved: submission_xgb_blend.csv')
print(f'Shape: {submission.shape}')
print('\nPrediction distribution:')
print(submission['disorder'].value_counts().sort_index())
print(f'\nSubmit if final OOF > 0.3860. Current: {final_oof_ba:.4f}')

Saved: submission_xgb_blend.csv
Shape: (8834, 1)

Prediction distribution:
disorder
0     392
1    1390
2     640
3    1572
4     243
5    1296
6    1083
7    1166
8     231
9     821
Name: count, dtype: int64

Submit if final OOF > 0.3860. Current: 0.3933
